# Orchestrating the LLM Training Loop

You've built the beautiful, complex machinery of a GPT model. You have a mountain of text data. Now what? This notebook is about building the conductor for our orchestra—the training loop that connects the data to the model and teaches it, note by note, to create a symphony of coherent text.

By the end of this notebook, you will be able to orchestrate the entire training process for a language model. You will understand how to feed data to the model in batches, calculate its performance, and use that feedback to systematically improve it. We will build the simple, powerful engine that drives all of the learning in a Large Language Model.

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import time

# Mock settings to simulate a real environment
VOCAB_SIZE = 50 # Size of our dictionary (e.g., number of unique characters)
BATCH_SIZE = 8  # How many independent sequences we process in parallel
BLOCK_SIZE = 32 # The maximum context length for predictions
LEARNING_RATE = 1e-3
MAX_ITERS = 100 # Total training iterations

### The Core Idea: Teaching a Baker to Bake

Imagine a novice baker trying to perfect a recipe for bread. They don't bake one giant loaf and decide if it's good or bad. Instead, they bake many small batches, learning from each one. This iterative process is exactly how we train a neural network.

The training loop is our baker's learning process, broken down into a few key steps:

1.  **Get a Batch of Ingredients**: The baker doesn't use all their flour at once. They take a small, manageable amount—a "batch." For us, this means grabbing a small chunk of our text data.
2.  **Bake the Bread (Forward Pass)**: The baker follows the current recipe to produce a loaf. Our model takes the input text batch and processes it to generate predictions for the next word at every position.
3.  **Taste Test (Calculate Loss)**: The baker tastes the bread and compares it to their ideal. "Is it too salty? Not fluffy enough?" This "error" is the **loss**. We compare the model's predictions to the actual next words in the text and calculate a numerical score of how wrong it was.
4.  **Figure Out What Went Wrong (Backward Pass)**: The baker thinks, "The bread was too salty because I added too much salt at the beginning." They trace the error back to the specific action that caused it. This is **backpropagation**, where we calculate how much each knob (parameter) in our model contributed to the final error. These calculated contributions are called **gradients**.
5.  **Adjust the Recipe (Optimizer Step)**: The baker decides, "Next time, I'll use a pinch less salt." They make a small adjustment to their recipe. Our **optimizer** uses the gradients to make small adjustments to all the model's parameters, nudging them in a direction that should reduce the error.
6.  **Clean the Counter (Zero Gradients)**: The baker cleans the workspace to start fresh for the next batch. We reset the gradients to zero so that the error calculation for the next batch isn't mixed with the old one.

We repeat this cycle hundreds, thousands, or millions of times. With each iteration, the baker's recipe—and our model's parameters—gets a little bit better.

In [ ]:
# A mock GPT model for demonstration purposes.
# In the previous notebook, we built a more complex version of this.
class MockGPT(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx):
        # idx is (Batch, Time) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, vocab_size)
        return logits

# --- The Training Loop ---
def train():
    # 1. Create a mock model and optimizer
    model = MockGPT(VOCAB_SIZE, BLOCK_SIZE)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    # 2. Create some dummy data
    full_data = torch.randint(0, VOCAB_SIZE, (1000,))

    # 3. The training loop itself
    for step in range(MAX_ITERS):
        # --- The core of one training step ---
        
        # A. Get a batch of data
        # We grab a random chunk of data to train on
        indices = torch.randint(0, len(full_data) - BLOCK_SIZE, (BATCH_SIZE,))
        input_batch = torch.stack([full_data[i:i+BLOCK_SIZE] for i in indices])
        target_batch = torch.stack([full_data[i+1:i+BLOCK_SIZE+1] for i in indices])

        # B. Forward pass: Ask the model for its predictions
        logits = model(input_batch)

        # C. Calculate loss: How wrong were the predictions?
        # We need to reshape for CrossEntropyLoss which expects (N, C)
        B, T, C = logits.shape
        loss = loss_fn(logits.view(B*T, C), target_batch.view(B*T))

        # D. Backward pass: Calculate gradients
        loss.backward()

        # E. Optimizer step: Update model parameters
        optimizer.step()
        
        # F. Zero gradients: Reset for the next iteration
        optimizer.zero_grad(set_to_none=True)
        # --- End of the core training step ---

        if step % 10 == 0:
            print(f"Step {step:4d}, Loss: {loss.item():.4f}")

    return model

### Walking Through the Code

Let's break down our `train` function, connecting it back to our baker analogy.

-   **`model = MockGPT(...)`**: We create an instance of our simple model. This is the baker's initial, unrefined recipe book.
-   **`optimizer = AdamW(...)`**: We hire a "recipe consultant" (the AdamW optimizer). We tell it which parameters it's in charge of (`model.parameters()`) and how aggressively it should suggest changes (the `lr`, or learning rate).
-   **`full_data = torch.randint(...)`**: We prepare our entire pantry of ingredients—a long sequence of random token IDs.
-   **`for step in range(MAX_ITERS):`**: This loop represents the baker's work day, where they bake one batch after another to practice.

Inside the loop, the magic happens:

1.  **`indices = torch.randint(...)` and `torch.stack([...])`**: This is our **`get_batch`** step. We randomly select `BATCH_SIZE` starting points in our `full_data` and stack them into two tensors: `input_batch` (the context) and `target_batch` (the expected next word for each context).
2.  **`logits = model(input_batch)`**: The **forward pass**. We give the ingredients (input batch) to the model and let it follow its current recipe to produce a result (`logits`, which are raw predictions).
3.  **`loss = loss_fn(...)`**: The **taste test**. We use the `CrossEntropyLoss` function to compare the model's `logits` against the `target_batch`. It outputs a single number, `loss`, representing how "bad" the model's predictions were. A high loss means the bread tastes terrible; a low loss means it's getting close to perfect.
4.  **`loss.backward()`**: Figuring out what went wrong. PyTorch automatically traces the calculations that led to the `loss` and computes the gradient for every single parameter in the model. It's like the baker instantly knowing that the salty taste came 70% from the salt shaker and 30% from the salty butter.
5.  **`optimizer.step()`**: Adjusting the recipe. The optimizer looks at the gradients (e.g., "too much saltiness came from this parameter") and nudges the parameters in the opposite direction.
6.  **`optimizer.zero_grad()`**: Cleaning the counter. We erase the old gradient information to ensure the next batch's feedback is calculated fresh.

The `print` statement simply gives us a status update, showing that the `loss` is (hopefully) decreasing over time.

In [ ]:
# Let's demonstrate the training loop in action.
# We expect to see the loss value generally decrease over the 100 steps.

print("Starting the training process...")
trained_model = train()
print("\nTraining complete!")

### Going Deeper: Why Not Just Load Everything Into RAM?

In our simple example, `full_data` was a small tensor that fit easily in memory. But what about real-world datasets? The text of Wikipedia, for example, is over 20 gigabytes. The full dataset used to train GPT-3 is 45 *terabytes*. You can't just load that into your computer's RAM.

This is where a clever technique used in `nanoGPT` comes in: **memory mapping**.

Instead of loading the entire file into RAM, you tell the operating system: "See this giant file on my hard drive? Just pretend it's an array in my program's memory." This is called a `memory-map`. When your code tries to access a part of the "array" (e.g., to get a batch), the OS transparently fetches just that small piece from the disk.

This gives you the best of both worlds: you can work with datasets that are far larger than your available RAM, while keeping your data-loading code clean and looking as if it's just slicing a normal array.

Let's build a simplified version of `nanoGPT`'s `get_batch` function that uses this principle.

In [ ]:
def setup_mock_dataset(filename, num_tokens):
    """Creates a large binary file of random tokens."""
    print(f"Creating a mock dataset file: {filename}")
    # Use 16-bit integers, just like nanoGPT
    mock_data = np.random.randint(0, VOCAB_SIZE, size=num_tokens, dtype=np.uint16)
    with open(filename, 'wb') as f:
        f.write(mock_data.tobytes())
    print("Mock dataset created.")

def get_batch_from_disk(data_file, block_size, batch_size):
    """
    Gets a batch of data by reading from a memory-mapped file on disk.
    This is much more memory-efficient for large datasets.
    """
    # Open the file in memory-map mode. 'r' is for read-only.
    # The OS handles loading chunks of the file as needed.
    data = np.memmap(data_file, dtype=np.uint16, mode='r')
    
    # The rest of the logic is the same as before!
    # We can treat `data` as if it were a normal numpy array.
    num_sequences = len(data) - block_size
    ix = np.random.randint(0, num_sequences, batch_size)
    
    # Accessing data[i:i+block_size] triggers the OS to read that chunk from disk.
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    
    return x, y

# --- Demonstration ---
DATA_FILE = 'train.bin'
# Create a dataset with 1 million tokens
setup_mock_dataset(DATA_FILE, 1_000_000)

print("\nFetching a batch from the memory-mapped file:")
inputs, targets = get_batch_from_disk(DATA_FILE, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE)
print("Input batch shape:", inputs.shape)
print("Target batch shape:", targets.shape)

# Clean up the dummy file
os.remove(DATA_FILE)

### Visualizing the Training Process

It can be hard to intuit what "decreasing the loss" really means. Let's visualize it. Instead of training a giant GPT, we'll train a tiny model to fit a simple line (`y = weight * x + bias`) to some data. The `weight` and `bias` are our model's "parameters."

The training loop's job is to find the best `weight` and `bias` to make the line pass as closely as possible through the data points. Watch as the orange line (our model's prediction) gets pulled towards the blue data points with each training step, and see how the loss (the error) drops accordingly.

In [ ]:
# --- Setup for visualization ---
# 1. Generate some noisy linear data
true_weight = 2.5
true_bias = 0.5
X = torch.randn(200, 1) * 10
y = true_weight * X + true_bias + torch.randn(200, 1) * 2 # Add some noise

# 2. A simple linear regression model
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1) # One input feature, one output feature

    def forward(self, x):
        return self.linear(x)

# 3. Setup the training components
model = LinearModel()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss() # Mean Squared Error is good for regression

# --- Animation function ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
losses = []

def animate(frame):
    # One step of the training loop
    # Forward pass
    predictions = model(X)
    loss = loss_fn(predictions, y)
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    # Update plots
    ax1.clear()
    ax1.scatter(X, y, label='Data', alpha=0.5)
    ax1.plot(X, predictions.detach().numpy(), color='orange', linewidth=3, label='Model Prediction')
    ax1.set_title(f"Step: {frame+1}")
    ax1.set_xlabel("Input (X)")
    ax1.set_ylabel("Target (y)")
    ax1.legend()
    
    ax2.clear()
    ax2.plot(losses)
    ax2.set_title("Loss Over Time")
    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Loss")

# Run the animation
# NOTE: This can be slow in some notebook environments.
# We'll use a simpler loop with clear_output for robustness.
print("Visualizing 50 steps of training. Please wait...")

for i in range(50):
    animate(i)
    # Use display and clear_output for a smoother experience in Jupyter
    display(fig)
    clear_output(wait=True)
    if i < 49: # Don't sleep on the last frame
        time.sleep(0.05)

plt.close(fig) # Close the final figure to avoid double printing
print("Visualization complete.")
# Re-display the final state
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
animate(49)
plt.show()

### Summary

In this chapter, we built the heart of the training process: the training loop. This orchestration is what allows a model to learn from data. Without it, the most sophisticated model architecture is just a static object, incapable of improvement.

**What We Built:**
- A complete, step-by-step training loop that performs a forward pass, calculates loss, runs a backward pass, and updates model weights.
- An understanding of batching and how to handle huge datasets efficiently using memory-mapping.
- A clear, visual intuition for how an optimizer uses loss to guide a model's parameters towards a better solution.

**Key Takeaways:**
- Training is an iterative cycle: `get batch -> forward -> loss -> backward -> optimize`.
- The `loss` is the critical signal that tells the model how well it's doing.
- The `optimizer` is the mechanism that translates the `loss` signal into concrete parameter updates.
- Real-world training requires careful data handling to manage memory when datasets are larger than RAM.

**What's Next?**
We now have all the conceptual pieces: a dataset, a model architecture, and a training loop to teach the model. Assuming we've run this loop for thousands of iterations and have saved our `trained_model`, what can we do with it? In the next notebook, **"Generating Text with a Trained GPT,"** we'll finally see the fruits of our labor and teach our model to write new text, character by character.